In [27]:
# =========================
# 📂 CARGAR DATA
# =========================
df = pd.read_csv('../data/processed/costos_final2.csv')

# =========================
# 💰 CREAR COSTO TOTAL
# =========================
df['costo_total'] = df['costo_material'] + df['costo_mano_obra']

# =========================
# 📊 VALIDACIÓN
# =========================
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 4971 entries, 0 to 4970
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   proyecto               4971 non-null   int64  
 1   nombre_proyecto        4971 non-null   str    
 2   tipo_proyecto          4971 non-null   str    
 3   fase                   4971 non-null   str    
 4   actividad              4971 non-null   str    
 5   material               4971 non-null   str    
 6   unidad_medida          4971 non-null   str    
 7   cantidad_material      4971 non-null   int64  
 8   costo_unitario         4971 non-null   int64  
 9   costo_material         4971 non-null   int64  
 10  dias_trabajados        4971 non-null   int64  
 11  duracion_estimada      4971 non-null   int64  
 12  duracion_real          4971 non-null   int64  
 13  retraso_dias           4971 non-null   int64  
 14  costo_mano_obra        4971 non-null   float64
 15  costo_estimado 

In [28]:
df['fecha_inicio'] = pd.to_datetime(df['fecha_inicio'], errors='coerce').dt.date
df['fecha_fin'] = pd.to_datetime(df['fecha_fin'], errors='coerce').dt.date

In [34]:
# convertir otra vez a datetime para trabajar
from turtle import pd
import pandas as pd


df['fecha_inicio'] = pd.to_datetime(df['fecha_inicio'])
df['fecha_fin'] = pd.to_datetime(df['fecha_fin'])

# rango permitido
fecha_min = pd.to_datetime("2025-01-01")
fecha_max = pd.to_datetime("2026-04-30")

# orden lógico de fases
orden_fases = {
    'cimentacion': 1,
    'estructura': 2,
    'instalaciones': 3,
    'obra gris': 4,
    'acabados': 5,
    'equipamiento': 6,
    'carpinteria y cerramiento': 7
}

# recorrer por proyecto
for proyecto in df['proyecto'].unique():
    
    mask = df['proyecto'] == proyecto
    df_proj = df.loc[mask].copy()

    # 📅 inicio aleatorio del proyecto (dentro del rango)
    inicio_proyecto = fecha_min + pd.to_timedelta(np.random.randint(0, 365), unit='D')

    df_proj['orden'] = df_proj['fase'].map(orden_fases)
    df_proj = df_proj.sort_values(by='orden')

    fechas_ref = {}

    for idx in df_proj.index:

        fase = df.loc[idx, 'fase']

        # duración real o generada
        duracion = df.loc[idx, 'duracion_real']
        if pd.isna(duracion):
            duracion = np.random.randint(5, 20)

 # 🎯 lógica constructiva segura
        if fase == 'cimentacion':
            inicio = inicio_proyecto

        elif fase == 'estructura':
            inicio = fechas_ref.get('cimentacion', inicio_proyecto) + pd.Timedelta(days=10)

        elif fase in ['instalaciones', 'obra gris']:
            base = fechas_ref.get('estructura', inicio_proyecto)
            inicio = base + pd.Timedelta(days=duracion // 2)

        elif fase == 'acabados':
            base = fechas_ref.get('obra gris', inicio_proyecto)
            inicio = base + pd.Timedelta(days=duracion // 2)

        elif fase in ['equipamiento', 'carpinteria y cerramiento']:
            base = fechas_ref.get('acabados', inicio_proyecto)
            inicio = base + pd.Timedelta(days=duracion // 2)

        else:
            inicio = inicio_proyecto

        fin = inicio + pd.Timedelta(days=duracion)

        # 🚫 evitar que se salga del rango
        if fin > fecha_max:
            fin = fecha_max
            inicio = fin - pd.Timedelta(days=duracion)

        df.loc[idx, 'fecha_inicio'] = inicio
        df.loc[idx, 'fecha_fin'] = fin

        fechas_ref[fase] = inicio

In [41]:
df.to_csv('../data/costos_final_limpio.csv', index=False)

In [42]:
df['costo_total'] = df['costo_total'].round(0).astype(int)

In [38]:
columnas = [
    'costo_material',
    'duracion_estimada',
    'duracion_real',
    'retraso_dias',
    'costo_mano_obra',
    'costo_estimado',
    'costo_real',
    'diferencia_costo'
]

df[columnas] = df[columnas].round(0).astype('Int64')

In [40]:
df['porcentaje_desviacion'] = df['porcentaje_desviacion'].round(1)

df['porcentaje_desviacion'] = df['porcentaje_desviacion'].map(lambda x: f"{x:.1f}%")